In [ ]:
import psutil

ram_gb = psutil.virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 13.6 gigabytes of available RAM

Not using a high-RAM runtime


In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


In [ ]:
# ✅ One-time setup for dependencies
!pip install zarr numcodecs tqdm pandas openpyxl


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.4/205.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 2.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount("/content/drive")



Mounted at /content/drive


In [ ]:
%cd "/content/drive/MyDrive/Colab Notebooks/LLM Project"


/content/drive/MyDrive/Colab Notebooks/LLM Project


In [ ]:
!for file in *.zarr.zip; do echo "📂 Unzipping $file..."; unzip -q "$file"; done


📂 Unzipping 2018.zarr.zip...


In [ ]:
import os
import zarr
import numpy as np
import tensorflow as tf
import gc
import pandas as pd
from tqdm import tqdm
from scipy.linalg import solve
from numcodecs import Blosc

class EnvironmentalInputOutputModelTF:
    def __init__(self, data_path):
        print(f"Opening Zarr folder: {data_path}")
        self.store = zarr.open(data_path, mode='r')
        self._extract_data()

    def _extract_data(self):
        print("Loading T, Y, Q arrays...")
        self.transactions = self.store["T"][:]   # (189,163,189,163)
        self.final_demand = self.store["Y"][:]   # (189,163,189)
        self.environmental_factors = self.store["Q"][:]  # (18,189,163)

        self.num_regions = 189
        self.num_sectors = 163
        self.total_units = self.num_regions * self.num_sectors
        self.num_env_indicators = self.environmental_factors.shape[0]

        print("Reshaping and converting to TensorFlow tensors...")
        Z = self.transactions.reshape(self.total_units, self.total_units)
        Y = self.final_demand.reshape(self.total_units, self.num_regions).sum(axis=1)
        Q = self.environmental_factors.reshape(self.num_env_indicators, self.total_units)

        self.Z = tf.convert_to_tensor(Z, dtype=tf.float32)
        self.Y = tf.convert_to_tensor(Y, dtype=tf.float32)
        self.Q = tf.convert_to_tensor(Q, dtype=tf.float32)
        self.Y_full = tf.convert_to_tensor(self.final_demand.reshape(self.total_units, self.num_regions), dtype=tf.float32)

        del self.transactions, self.final_demand, self.environmental_factors
        gc.collect()

    def calculate_leontief_inverse(self):
        print("Calculating Leontief inverse...")
        x = tf.reduce_sum(self.Z, axis=1) + self.Y
        x_safe = tf.where(x <= 0, tf.constant(1.0, dtype=tf.float32), x)
        A = self.Z / tf.reshape(x_safe, (-1, 1))

        I = tf.eye(self.total_units, dtype=tf.float32)
        B = I - A

        B_np = B.numpy()
        L_np = np.linalg.inv(B_np)

        del A, B, I
        gc.collect()

        return tf.convert_to_tensor(L_np, dtype=tf.float32), x_safe

    def compute_environmental_impact(self, output_path, batch=512):
        print("Computing environmental impact matrix D = S * L_y")
        L_y, x = self.calculate_leontief_inverse()

        S = self.Q / tf.reshape(x, (1, -1))
        S = tf.where(tf.math.is_finite(S), S, 0)

        print(f"Writing output to: {output_path}")
        zarr_out = zarr.open(output_path, mode="w", zarr_version=2)
        dataset = zarr_out.create_dataset(
            name="Environmental_Impact_3D",
            shape=(self.num_env_indicators, self.total_units, self.total_units),
            chunks=(1, batch, self.total_units),
            dtype="float32",
            compressor=Blosc(cname="zstd", clevel=5),
            overwrite=True
        )

        S_np = S.numpy()
        L_y_np = L_y @ tf.reshape(self.Y, (-1, 1))
        L_y_np = tf.reshape(L_y_np, (-1,)).numpy()

        for env_idx in tqdm(range(self.num_env_indicators), desc="Writing impact", unit="indicator"):
            impact_vector = S_np[env_idx]
            for start_idx in range(0, self.total_units, batch):
                end_idx = min(start_idx + batch, self.total_units)
                block = impact_vector[start_idx:end_idx, None] * L_y_np[None, :]
                dataset[env_idx, start_idx:end_idx, :] = block.astype(np.float32)

        print("Finished saving impact matrix.")

        del L_y_np, S_np, L_y, S, x
        gc.collect()

    def compute_additional_metrics(self):
        x = tf.reduce_sum(self.Z, axis=1) + self.Y
        Q = self.Q

        env_intensity = Q / tf.reshape(x, (1, -1))
        env_intensity = tf.where(tf.math.is_finite(env_intensity), env_intensity, 0)
        reshaped = tf.reshape(env_intensity, (self.num_env_indicators, self.num_regions, self.num_sectors))
        total_region_intensity = tf.reduce_sum(reshaped, axis=2)

        intensity_df = pd.DataFrame(
            env_intensity.numpy().reshape(self.num_env_indicators, self.total_units).T,
            columns=[f"indicator_{i}" for i in range(self.num_env_indicators)]
        )
        region_agg_df = pd.DataFrame(
            total_region_intensity.numpy().T,
            columns=[f"indicator_{i}" for i in range(self.num_env_indicators)]
        )

        return intensity_df, region_agg_df

def run_processing_for_year(year, base_dir):
    input_path = os.path.join(base_dir, f"{year}.zarr")
    output_path = os.path.join(base_dir, f"Environmental_Impact_{year}.zarr")

    print(f"\nStarting year: {year}")
    print(f"Looking for: {input_path}")

    if not os.path.isdir(input_path):
        print(f"Missing folder: {input_path}, skipping.")
        return

    model = EnvironmentalInputOutputModelTF(input_path)
    model.compute_environmental_impact(output_path)
    print(f"Saved result to: {output_path}")

    z = zarr.open(output_path, mode='r')
    shape = z["Environmental_Impact_3D"].shape
    print(f"Shape of D matrix for {year}: {shape}")

    del model
    gc.collect()

if __name__ == "__main__":
    base_directory = "/content/drive/MyDrive/Colab Notebooks/LLM Project"
    years_to_process = [2018, 2019, 2020, 2021, 2022]

    for yr in years_to_process:
        run_processing_for_year(yr, base_directory)



Starting year: 2018
Looking for: /content/drive/MyDrive/Colab Notebooks/LLM Project/2018.zarr
Opening Zarr folder: /content/drive/MyDrive/Colab Notebooks/LLM Project/2018.zarr
Loading T, Y, Q arrays...
Reshaping and converting to TensorFlow tensors...
Computing environmental impact matrix D = S * L_y
Calculating Leontief inverse...
Writing output to: /content/drive/MyDrive/Colab Notebooks/LLM Project/Environmental_Impact_2018.zarr


<ipython-input-9-ef9c6b223bdf>:67: DeprecationWarning: Use Group.create_array instead.
  dataset = zarr_out.create_dataset(
Writing impact: 100%|██████████| 19/19 [14:47<00:00, 46.69s/indicator]


Finished saving impact matrix.
Saved result to: /content/drive/MyDrive/Colab Notebooks/LLM Project/Environmental_Impact_2018.zarr
Shape of D matrix for 2018: (19, 30807, 30807)

Starting year: 2019
Looking for: /content/drive/MyDrive/Colab Notebooks/LLM Project/2019.zarr
Opening Zarr folder: /content/drive/MyDrive/Colab Notebooks/LLM Project/2019.zarr
Loading T, Y, Q arrays...
Reshaping and converting to TensorFlow tensors...
Computing environmental impact matrix D = S * L_y
Calculating Leontief inverse...


<ipython-input-9-ef9c6b223bdf>:67: DeprecationWarning: Use Group.create_array instead.
  dataset = zarr_out.create_dataset(


Writing output to: /content/drive/MyDrive/Colab Notebooks/LLM Project/Environmental_Impact_2019.zarr


...

Finished saving impact matrix.
Saved result to: /content/drive/MyDrive/Colab Notebooks/LLM Project/Environmental_Impact_2021.zarr
Shape of D matrix for 2021: (19, 30807, 30807)

Starting year: 2022
Looking for: /content/drive/MyDrive/Colab Notebooks/LLM Project/2022.zarr
Opening Zarr folder: /content/drive/MyDrive/Colab Notebooks/LLM Project/2022.zarr
Loading T, Y, Q arrays...
Reshaping and converting to TensorFlow tensors...
Computing environmental impact matrix D = S * L_y
Calculating Leontief inverse...


<ipython-input-9-ef9c6b223bdf>:67: DeprecationWarning: Use Group.create_array instead.
  dataset = zarr_out.create_dataset(


Writing output to: /content/drive/MyDrive/Colab Notebooks/LLM Project/Environmental_Impact_2022.zarr


Writing impact: 100%|██████████| 19/19 [12:19<00:00, 38.95s/indicator]


Finished saving impact matrix.
Saved result to: /content/drive/MyDrive/Colab Notebooks/LLM Project/Environmental_Impact_2022.zarr
Shape of D matrix for 2022: (19, 30807, 30807)
